In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torchaudio

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
import pickle
import random
from torch.utils.data import Dataset, DataLoader

In [3]:
import os
os.environ['TORCH_USE_CUDA_DSA'] = '1'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
print("Set TORCH_USE_CUDA_DSA=1 and CUDA_LAUNCH_BLOCKING=1. Please restart the Colab runtime and then re-run all cells.")

Set TORCH_USE_CUDA_DSA=1 and CUDA_LAUNCH_BLOCKING=1. Please restart the Colab runtime and then re-run all cells.


In [4]:
with open("/content/drive/MyDrive/final_features.pkl", "rb") as f:
    data = pickle.load(f)

# ensure labels are int
for item in data:
    item["label"] = int(item["label"])

print(set([item["label"] for item in data]))  # must be {0,1}

{0, 1}


In [5]:
class ContrastiveDataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.class_data = {0: [], 1: []}

        for item in data:
            self.class_data[item["label"]].append(item)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item1 = self.data[idx]
        label1 = item1["label"]

        if random.random() < 0.5:
            item2 = random.choice(self.class_data[label1])
            pair_label = 1
        else:
            item2 = random.choice(self.class_data[1 - label1])
            pair_label = 0

        return item1, item2, pair_label

In [6]:
class ConformerEncoder(nn.Module):
    def __init__(self, input_dim=80, d_model=128):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        self.conformer = torchaudio.models.Conformer(
            input_dim=d_model,
            num_heads=4,
            ffn_dim=256,
            num_layers=2,
            depthwise_conv_kernel_size=3 # Reverting to smaller kernel size to fix CUDA error
        )

    def forward(self, x):
        x = self.input_proj(x)

        lengths = torch.full(
            (x.shape[0],),
            x.shape[1],
            dtype=torch.long,
            device=x.device
        )

        out, _ = self.conformer(x, lengths)
        return out.mean(dim=1)

In [7]:
class CrossAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, 4, batch_first=True)

    def forward(self, q, k, v):
        out, _ = self.attn(q, k, v)
        return out

In [8]:
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.audio_encoder = ConformerEncoder()
        self.text_proj = nn.Linear(768, 128)

        self.cross_ta = CrossAttention(128)
        self.cross_at = CrossAttention(128)

        self.fusion = nn.Linear(256, 128)

        self.final = nn.Sequential(
            nn.Linear(128 + 8, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        self.classifier = nn.Linear(32, 2)

    def forward(self, mel, bert, behavior):

        A = self.audio_encoder(mel).unsqueeze(1)
        T = self.text_proj(bert).unsqueeze(1)

        Z_ta = self.cross_ta(T, A, A)
        Z_at = self.cross_at(A, T, T)

        Z = torch.cat([Z_ta.squeeze(1), Z_at.squeeze(1)], dim=1)
        Z = self.fusion(Z)

        Z_final = torch.cat([Z, behavior], dim=1)

        h = self.final(Z_final)
        logits = self.classifier(h)

        return h, logits

In [9]:
def prepare_batch(batch):

    x1, x2, pair_label = zip(*batch)

    def process(items):

        mel_list, bert, behavior, labels = [], [], [], []

        for item in items:
            f = item["features"]

            mel_list.append(torch.tensor(f["mel"].T, dtype=torch.float32))
            bert.append(f["bert"])

            behavior.append([
                f["pause_count"],
                f["avg_pause"],
                f["speech_rate"],
                f["filler_count"],
                f["repetition"],
                f["turn_length"],
                f["lexical_diversity"],
                f["sentence_length"]
            ])

            labels.append(int(item["label"]))

        mel = torch.nn.utils.rnn.pad_sequence(mel_list, batch_first=True)

        return (
            mel,
            torch.tensor(np.array(bert), dtype=torch.float32),
            torch.tensor(np.array(behavior), dtype=torch.float32),
            torch.LongTensor(labels)
        )

    return process(x1), process(x2), torch.tensor(pair_label, dtype=torch.float32)

In [10]:
def contrastive_loss(h1, h2, label, margin=1.0):
    dist = F.pairwise_distance(h1, h2)
    return torch.mean(
        label * dist**2 +
        (1 - label) * torch.clamp(margin - dist, min=0.0)**2
    )


def prototype_loss(h, y):
    loss = 0
    for i in range(len(h)):
        same = h[y == y[i]]
        proto = same.mean(dim=0)
        loss += torch.norm(h[i] - proto)
    return loss / len(h)

In [11]:
 device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = ContrastiveDataset(data)
loader = DataLoader(dataset, batch_size=6, shuffle=True, collate_fn=prepare_batch)

model = MultiModalModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(15):

    total_loss = 0

    for (m1, b1, beh1, y1), (m2, b2, beh2, y2), pair_y in loader:

        m1, b1, beh1, y1 = m1.to(device), b1.to(device), beh1.to(device), y1.to(device)
        m2, b2, beh2, y2 = m2.to(device), b2.to(device), beh2.to(device), y2.to(device)
        pair_y = pair_y.to(device).clamp(0, 1);

        # SKIP BAD BATCHES (CRITICAL FIX)
        if m1.shape[1] == 0 or m2.shape[1] == 0:
            continue

        if torch.isnan(m1).any() or torch.isnan(b1).any() or torch.isnan(beh1).any():
            continue

        #  MAIN FIX — avoid single-class batch crash
        if len(torch.unique(y1)) < 2:
            continue

        h1, logits1 = model(m1, b1, beh1)
        h2, logits2 = model(m2, b2, beh2)

        y1 = y1.long().view(-1)

        loss_cls = F.cross_entropy(logits1, y1)
        loss_proto = prototype_loss(h1, y1)
        loss_con = contrastive_loss(h1, h2, pair_y)

        loss = loss_cls + 0.3 * loss_proto + 0.3 * loss_con

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# Path inside your Google Drive
save_path = "/content/drive/MyDrive/multimodal_model.pth"

torch.save(model.state_dict(), save_path)

print("Model saved to Google Drive!")

Epoch 0, Loss: 67114.1609
Epoch 1, Loss: 10208.6977
Epoch 2, Loss: 4756.0006
Epoch 3, Loss: 1916.0525
Epoch 4, Loss: 987.9330
Epoch 5, Loss: 630.9409
Epoch 6, Loss: 515.1029
Epoch 7, Loss: 485.9023
Epoch 8, Loss: 428.6991
Epoch 9, Loss: 372.3370
Epoch 10, Loss: 349.6735
Epoch 11, Loss: 335.7768
Epoch 12, Loss: 330.4132
Epoch 13, Loss: 312.7691
Epoch 14, Loss: 314.1987
Model saved to Google Drive!


FINAL TRAINING + FREEZE/UNFREEZE + ABLATION

In [12]:
MODE = "full"
# options:
# "full"
# "no_behavioral"
# "no_cross"
# "text_only"

In [13]:
def forward(self, mel, bert, behavior):

    A = self.audio_encoder(mel).unsqueeze(1)
    T = self.text_proj(bert).unsqueeze(1)

    if MODE == "no_cross":
        Z = torch.cat([A.squeeze(1), T.squeeze(1)], dim=1)

    else:
        Z_ta = self.cross_ta(T, A, A)
        Z_at = self.cross_at(A, T, T)
        Z = torch.cat([Z_ta.squeeze(1), Z_at.squeeze(1)], dim=1)

    Z = self.fusion(Z)

    if MODE == "no_behavioral":
        Z_final = Z
    elif MODE == "text_only":
        Z_final = T.squeeze(1)
    else:
        Z_final = torch.cat([Z, behavior], dim=1)

    h = self.final(Z_final)
    logits = self.classifier(h)

    return h, logits

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = ContrastiveDataset(data)
loader = DataLoader(dataset, batch_size=6, shuffle=True, collate_fn=prepare_batch)

model = MultiModalModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [15]:
# Freeze audio encoder
for param in model.audio_encoder.parameters():
    param.requires_grad = False

In [16]:
best_loss = float("inf")

for epoch in range(15):

    total_loss = 0

    #  UNFREEZE AFTER 3 EPOCHS
    if epoch == 3:
        print(" Unfreezing model")
        for param in model.parameters():
            param.requires_grad = True

    for (m1, b1, beh1, y1), (m2, b2, beh2, y2), pair_y in loader:

        m1, b1, beh1, y1 = m1.to(device), b1.to(device), beh1.to(device), y1.to(device)
        m2, b2, beh2, y2 = m2.to(device), b2.to(device), beh2.to(device), y2.to(device)
        pair_y = pair_y.to(device).clamp(0, 1)

        # Skip bad batches
        if m1.shape[1] == 0 or m2.shape[1] == 0:
            continue

        if torch.isnan(m1).any() or torch.isnan(b1).any() or torch.isnan(beh1).any():
            continue

        if len(torch.unique(y1)) < 2:
            continue

        h1, logits1 = model(m1, b1, beh1)
        h2, logits2 = model(m2, b2, beh2)

        y1 = y1.long().view(-1)

        loss_cls = F.cross_entropy(logits1, y1)
        loss_proto = prototype_loss(h1, y1)
        loss_con = contrastive_loss(h1, h2, pair_y)

        loss = loss_cls + 0.3 * loss_proto + 0.3 * loss_con

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

    # Save best model
    if total_loss < best_loss:
        best_loss = total_loss
        torch.save(model.state_dict(), f"/content/drive/MyDrive/best_model_{MODE}.pth")
        print("Best model saved")

Epoch 0, Loss: 16889.6534
Best model saved
Epoch 1, Loss: 2735.0648
Best model saved
Epoch 2, Loss: 825.2022
Best model saved
 Unfreezing model
Epoch 3, Loss: 513.5641
Best model saved
Epoch 4, Loss: 396.0872
Best model saved
Epoch 5, Loss: 370.5211
Best model saved
Epoch 6, Loss: 338.3816
Best model saved
Epoch 7, Loss: 314.2025
Best model saved
Epoch 8, Loss: 303.7226
Best model saved
Epoch 9, Loss: 297.0674
Best model saved
Epoch 10, Loss: 285.9675
Best model saved
Epoch 11, Loss: 275.3374
Best model saved
Epoch 12, Loss: 266.8016
Best model saved
Epoch 13, Loss: 315.5674
Epoch 14, Loss: 277.2844


In [17]:
MODE = "no_behavioral"
def forward(self, mel, bert, behavior):

    A = self.audio_encoder(mel).unsqueeze(1)
    T = self.text_proj(bert).unsqueeze(1)

    if MODE == "no_cross":
        Z = torch.cat([A.squeeze(1), T.squeeze(1)], dim=1)

    else:
        Z_ta = self.cross_ta(T, A, A)
        Z_at = self.cross_at(A, T, T)
        Z = torch.cat([Z_ta.squeeze(1), Z_at.squeeze(1)], dim=1)

    Z = self.fusion(Z)

    if MODE == "no_behavioral":
        Z_final = Z
    elif MODE == "text_only":
        Z_final = T.squeeze(1)
    else:
        Z_final = torch.cat([Z, behavior], dim=1)

    h = self.final(Z_final)
    logits = self.classifier(h)

    return h, logits



In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = ContrastiveDataset(data)
loader = DataLoader(dataset, batch_size=6, shuffle=True, collate_fn=prepare_batch)

model = MultiModalModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [19]:
# Freeze audio encoder
for param in model.audio_encoder.parameters():
    param.requires_grad = False

In [20]:
best_loss = float("inf")

for epoch in range(15):

    total_loss = 0

    #  UNFREEZE AFTER 3 EPOCHS
    if epoch == 3:
        print(" Unfreezing model")
        for param in model.parameters():
            param.requires_grad = True

    for (m1, b1, beh1, y1), (m2, b2, beh2, y2), pair_y in loader:

        m1, b1, beh1, y1 = m1.to(device), b1.to(device), beh1.to(device), y1.to(device)
        m2, b2, beh2, y2 = m2.to(device), b2.to(device), beh2.to(device), y2.to(device)
        pair_y = pair_y.to(device).clamp(0, 1)

        # Skip bad batches
        if m1.shape[1] == 0 or m2.shape[1] == 0:
            continue

        if torch.isnan(m1).any() or torch.isnan(b1).any() or torch.isnan(beh1).any():
            continue

        if len(torch.unique(y1)) < 2:
            continue

        h1, logits1 = model(m1, b1, beh1)
        h2, logits2 = model(m2, b2, beh2)

        y1 = y1.long().view(-1)

        loss_cls = F.cross_entropy(logits1, y1)
        loss_proto = prototype_loss(h1, y1)
        loss_con = contrastive_loss(h1, h2, pair_y)

        loss = loss_cls + 0.3 * loss_proto + 0.3 * loss_con

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

    #  Save best model
    if total_loss < best_loss:
        best_loss = total_loss
        torch.save(model.state_dict(), f"/content/drive/MyDrive/best_model_{MODE}.pth")
        print(" Best model saved")

Epoch 0, Loss: 87671.1838
 Best model saved
Epoch 1, Loss: 16464.1418
 Best model saved
Epoch 2, Loss: 7536.1937
 Best model saved
 Unfreezing model
Epoch 3, Loss: 3720.0568
 Best model saved
Epoch 4, Loss: 1712.4541
 Best model saved
Epoch 5, Loss: 1015.5124
 Best model saved
Epoch 6, Loss: 892.3488
 Best model saved
Epoch 7, Loss: 605.4246
 Best model saved
Epoch 8, Loss: 490.9062
 Best model saved
Epoch 9, Loss: 455.5477
 Best model saved
Epoch 10, Loss: 395.9501
 Best model saved
Epoch 11, Loss: 373.7319
 Best model saved
Epoch 12, Loss: 361.8034
 Best model saved
Epoch 13, Loss: 388.6551
Epoch 14, Loss: 325.7104
 Best model saved


In [21]:
MODE = "no_cross"
def forward(self, mel, bert, behavior):

    A = self.audio_encoder(mel).unsqueeze(1)
    T = self.text_proj(bert).unsqueeze(1)

    if MODE == "no_cross":
        Z = torch.cat([A.squeeze(1), T.squeeze(1)], dim=1)

    else:
        Z_ta = self.cross_ta(T, A, A)
        Z_at = self.cross_at(A, T, T)
        Z = torch.cat([Z_ta.squeeze(1), Z_at.squeeze(1)], dim=1)

    Z = self.fusion(Z)

    if MODE == "no_behavioral":
        Z_final = Z
    elif MODE == "text_only":
        Z_final = T.squeeze(1)
    else:
        Z_final = torch.cat([Z, behavior], dim=1)

    h = self.final(Z_final)
    logits = self.classifier(h)

    return h, logits



In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = ContrastiveDataset(data)
loader = DataLoader(dataset, batch_size=6, shuffle=True, collate_fn=prepare_batch)

model = MultiModalModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [23]:
# Freeze audio encoder
for param in model.audio_encoder.parameters():
    param.requires_grad = False

In [24]:
best_loss = float("inf")

for epoch in range(15):

    total_loss = 0

    # UNFREEZE AFTER 3 EPOCHS
    if epoch == 3:
        print(" Unfreezing model")
        for param in model.parameters():
            param.requires_grad = True

    for (m1, b1, beh1, y1), (m2, b2, beh2, y2), pair_y in loader:

        m1, b1, beh1, y1 = m1.to(device), b1.to(device), beh1.to(device), y1.to(device)
        m2, b2, beh2, y2 = m2.to(device), b2.to(device), beh2.to(device), y2.to(device)
        pair_y = pair_y.to(device).clamp(0, 1)

        # Skip bad batches
        if m1.shape[1] == 0 or m2.shape[1] == 0:
            continue

        if torch.isnan(m1).any() or torch.isnan(b1).any() or torch.isnan(beh1).any():
            continue

        if len(torch.unique(y1)) < 2:
            continue

        h1, logits1 = model(m1, b1, beh1)
        h2, logits2 = model(m2, b2, beh2)

        y1 = y1.long().view(-1)

        loss_cls = F.cross_entropy(logits1, y1)
        loss_proto = prototype_loss(h1, y1)
        loss_con = contrastive_loss(h1, h2, pair_y)

        loss = loss_cls + 0.3 * loss_proto + 0.3 * loss_con

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

    #  Save best model
    if total_loss < best_loss:
        best_loss = total_loss
        torch.save(model.state_dict(), f"/content/drive/MyDrive/best_model_{MODE}.pth")
        print(" Best model saved")

Epoch 0, Loss: 34836.8611
 Best model saved
Epoch 1, Loss: 5820.5886
 Best model saved
Epoch 2, Loss: 2895.4390
 Best model saved
 Unfreezing model
Epoch 3, Loss: 952.1746
 Best model saved
Epoch 4, Loss: 623.8674
 Best model saved
Epoch 5, Loss: 477.0594
 Best model saved
Epoch 6, Loss: 418.1333
 Best model saved
Epoch 7, Loss: 369.8451
 Best model saved
Epoch 8, Loss: 355.3662
 Best model saved
Epoch 9, Loss: 353.3117
 Best model saved
Epoch 10, Loss: 355.5561
Epoch 11, Loss: 301.7603
 Best model saved
Epoch 12, Loss: 291.6276
 Best model saved
Epoch 13, Loss: 362.8413
Epoch 14, Loss: 387.1526


In [25]:
MODE = "text_only"
def forward(self, mel, bert, behavior):

    A = self.audio_encoder(mel).unsqueeze(1)
    T = self.text_proj(bert).unsqueeze(1)

    if MODE == "no_cross":
        Z = torch.cat([A.squeeze(1), T.squeeze(1)], dim=1)

    else:
        Z_ta = self.cross_ta(T, A, A)
        Z_at = self.cross_at(A, T, T)
        Z = torch.cat([Z_ta.squeeze(1), Z_at.squeeze(1)], dim=1)

    Z = self.fusion(Z)

    if MODE == "no_behavioral":
        Z_final = Z
    elif MODE == "text_only":
        Z_final = T.squeeze(1)
    else:
        Z_final = torch.cat([Z, behavior], dim=1)

    h = self.final(Z_final)
    logits = self.classifier(h)

    return h, logits


In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = ContrastiveDataset(data)
loader = DataLoader(dataset, batch_size=6, shuffle=True, collate_fn=prepare_batch)

model = MultiModalModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [27]:
# Freeze audio encoder
for param in model.audio_encoder.parameters():
    param.requires_grad = False

In [28]:
best_loss = float("inf")

for epoch in range(15):

    total_loss = 0

    #  UNFREEZE AFTER 3 EPOCHS
    if epoch == 3:
        print(" Unfreezing model")
        for param in model.parameters():
            param.requires_grad = True

    for (m1, b1, beh1, y1), (m2, b2, beh2, y2), pair_y in loader:

        m1, b1, beh1, y1 = m1.to(device), b1.to(device), beh1.to(device), y1.to(device)
        m2, b2, beh2, y2 = m2.to(device), b2.to(device), beh2.to(device), y2.to(device)
        pair_y = pair_y.to(device).clamp(0, 1)

        # Skip bad batches
        if m1.shape[1] == 0 or m2.shape[1] == 0:
            continue

        if torch.isnan(m1).any() or torch.isnan(b1).any() or torch.isnan(beh1).any():
            continue

        if len(torch.unique(y1)) < 2:
            continue

        h1, logits1 = model(m1, b1, beh1)
        h2, logits2 = model(m2, b2, beh2)

        y1 = y1.long().view(-1)

        loss_cls = F.cross_entropy(logits1, y1)
        loss_proto = prototype_loss(h1, y1)
        loss_con = contrastive_loss(h1, h2, pair_y)

        loss = loss_cls + 0.3 * loss_proto + 0.3 * loss_con

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

    # Save best model
    if total_loss < best_loss:
        best_loss = total_loss
        torch.save(model.state_dict(), f"/content/drive/MyDrive/best_model_{MODE}.pth")
        print("Best model saved")

Epoch 0, Loss: 21887.3110
Best model saved
Epoch 1, Loss: 3263.2501
Best model saved
Epoch 2, Loss: 1124.4800
Best model saved
 Unfreezing model
Epoch 3, Loss: 473.9395
Best model saved
Epoch 4, Loss: 477.9728
Epoch 5, Loss: 351.1190
Best model saved
Epoch 6, Loss: 370.2321
Epoch 7, Loss: 585.3584
Epoch 8, Loss: 343.3465
Best model saved
Epoch 9, Loss: 312.1783
Best model saved
Epoch 10, Loss: 309.8617
Best model saved
Epoch 11, Loss: 307.4026
Best model saved
Epoch 12, Loss: 309.1066
Epoch 13, Loss: 307.0973
Best model saved
Epoch 14, Loss: 303.0717
Best model saved


Evaluation

In [29]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

model.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():

    for (m1, b1, beh1, y1), _, _ in loader:

        m1, b1, beh1 = m1.to(device), b1.to(device), beh1.to(device)

        if m1.shape[1] == 0:
            continue

        h, logits = model(m1, b1, beh1)

        probs = torch.softmax(logits, dim=1)[:,1]
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y1.numpy())

In [30]:
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("AUC:", roc_auc_score(all_labels, all_probs))

Accuracy: 0.7749846719803801
F1: 0.8732297063903282
AUC: 0.8840822030317663


In [31]:
torch.save(model.state_dict(), "/content/drive/MyDrive/final_model.pth")

In [32]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def evaluate_model(model, loader, device):

    model.eval()

    all_preds = []
    all_probs = []
    all_labels = []
    all_paths = []

    with torch.no_grad():

        for (m1, b1, beh1, y1), _, _ in loader:

            m1, b1, beh1 = m1.to(device), b1.to(device), beh1.to(device)

            if m1.shape[1] == 0:
                continue

            if torch.isnan(m1).any() or torch.isnan(b1).any() or torch.isnan(beh1).any():
                continue

            h, logits = model(m1, b1, beh1)

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y1.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    return acc, f1, auc, all_preds, all_labels

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODES = ["full", "no_behavioral", "no_cross", "text_only"]

results = []

for mode in MODES:

    print(f"\n Evaluating: {mode}")

    MODE = mode  # set global mode

    model = MultiModalModel().to(device)
    model.load_state_dict(torch.load(f"/content/drive/MyDrive/best_model_{mode}.pth"))

    acc, f1, auc, preds, labels = evaluate_model(model, loader, device)

    results.append({
        "Model": mode,
        "Accuracy": acc,
        "F1": f1,
        "AUC": auc
    })


 Evaluating: full

 Evaluating: no_behavioral

 Evaluating: no_cross

 Evaluating: text_only


In [34]:
import pandas as pd

df = pd.DataFrame(results)

print("\n📊 FINAL RESULTS:")
print(df)

df.to_csv("/content/drive/MyDrive/final_results.csv", index=False)


📊 FINAL RESULTS:
           Model  Accuracy       F1       AUC
0           full  0.836297  0.90173  0.911897
1  no_behavioral  0.774985  0.87323  0.788411
2       no_cross  0.774985  0.87323  0.850473
3      text_only  0.774985  0.87323  0.883960


In [35]:
def task_wise_analysis(data, preds, labels):

    tasks = {
        "cookie": [],
        "sentence": [],
        "fluency": []
    }

    for item, pred, label in zip(data, preds, labels):

        path = item["audio"].lower()

        if "cookie" in path:
            tasks["cookie"].append((pred, label))
        elif "sentence" in path:
            tasks["sentence"].append((pred, label))
        elif "fluency" in path:
            tasks["fluency"].append((pred, label))

    for task in tasks:

        if len(tasks[task]) == 0:
            continue

        p = [x[0] for x in tasks[task]]
        l = [x[1] for x in tasks[task]]

        print(f"\n🔹 {task.upper()}")
        print("Accuracy:", accuracy_score(l, p))
        print("F1:", f1_score(l, p))

In [36]:
MODE = "full"

model = MultiModalModel().to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/best_model_full.pth"))

acc, f1, auc, preds, labels = evaluate_model(model, loader, device)

task_wise_analysis(data, preds, labels)


🔹 COOKIE
Accuracy: 0.8407294832826747
F1: 0.9051412020275162

🔹 SENTENCE
Accuracy: 0.8465679676985195
F1: 0.9074675324675324

🔹 FLUENCY
Accuracy: 0.8226544622425629
F1: 0.89198606271777


In [37]:

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, preds)
print("\nConfusion Matrix:\n", cm)


Confusion Matrix:
 [[ 282  452]
 [  79 2449]]
